# ⚙️ Hybrid Search & Cross-Encoder Reranking: Two-Stage Retrieval

**My Objective:** I am engineering a two-stage hybrid retrieval pipeline to eliminate the blind spots of single-method search. Dense vector embeddings fail on exact identifiers (e.g., `SKU-EL-004`, coupon codes), while sparse keyword search (BM25) fails on paraphrased queries (e.g., "smashed" vs. "damaged"). I am combining BM25 and dense vectors via Reciprocal Rank Fusion (RRF), applying a cross-encoder reranker (`Xenova/ms-marco-MiniLM-L-6-v2`) to maximize precision, calibrating a relevance gate to block out-of-scope queries, and deploying server-side hybrid search in Qdrant.

In [ ]:
import os
import sys
from pathlib import Path

# Secure, dynamic path resolution
WORK_REPO = Path(os.getenv("WORK_REPO", Path.cwd()))
for folder in ("vectordbs", "module04"):
    if (WORK_REPO / folder).exists():
        sys.path.insert(0, str(WORK_REPO / folder))
        break
else:
    sys.path.insert(0, str(WORK_REPO))

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv(usecwd=True))

import hb_utils as HB
import hb_queries as Q

rs = {c: HB.Retriever(c) for c in HB.CORPORA}
for c, r in rs.items():
    print(f"{c:<9} {len(r.chunks)} documents, e.g. {r.chunks[0].source}  [{r.chunks[0].doc_id}]")
print(len(Q.CASES), "test questions:", sum(k == "exact" for *_, k in Q.CASES), "exact,",
      sum(k == "meaning" for *_, k in Q.CASES), "meaning")

## 1. BM25 Tokenization Mechanics
Before sparse scoring occurs, text is normalized: lowercased, stripped of stop words, stemmed (e.g., `shoes` → `shoe`), and tokenized so compound product identifiers like `sku-fw-002` are preserved whole alongside their sub-tokens.

In [ ]:
for text in ["What is the price of SKU-FW-002 shoes?", "HDFC no-cost EMI", "the parcel arrived smashed"]:
    print(f"{text!r:<44} -> {HB.tokenize(text)}")

## 2. Failure Modes: Sparse vs. Dense Retrieval
Testing both engines against exact identifiers (`SKU-EL-004`) and semantic paraphrases (`the parcel arrived smashed`) exposes their complementary blind spots. Neither method alone is sufficient for production support workloads.

In [ ]:
def show(hits):
    for h in hits:
        print(f"   {h.score:8.3f}  {h.chunk.source}  [{h.chunk.doc_id}]")

for question, corpus in [("SKU-EL-004", "products"), ("the parcel arrived smashed", "help")]:
    r = rs[corpus]
    print(f'\n"{question}"  ({corpus})')
    print(" keyword:"); show(r.keyword(question))
    print(" vectors:"); show(r.dense(question))

## 3. BM25 Score Explainability
BM25 scores documents by combining Inverse Document Frequency (rare terms like `hdfc` carry higher weight), saturated term frequency (`k1`), and document length normalization (`b`).

In [ ]:
r = rs["help"]
q = "HDFC no-cost EMI"
for h in r.keyword(q, 2):
    print(f"{h.score:6.2f}  {h.chunk.source}")
    for w, idf, tf, part in r.bm25.explain(q, r.keys.index(h.key)):
        if tf:
            print(f"        {w:<10} idf {idf:4.2f}  x{tf}  -> +{part:.2f}")

## 4. Reciprocal Rank Fusion (RRF)
Because BM25 scores (0 to 20+) and cosine similarities (0 to 1) operate on incompatible scales, raw addition is invalid. RRF merges both rankings by summing `1 / (60 + rank)` from each list, promoting documents that rank highly across both engines without requiring score normalization.

In [ ]:
r, q = rs["products"], "PowerCell"
kw, dn = r.keyword_ranking(q, 20), r.dense_ranking(q, 20)
print(f"{'row':<22}{'keyword rank':>13}{'vector rank':>12}{'RRF':>9}")
for key, s in HB.rrf([kw, dn])[:6]:
    a = kw.index(key) + 1 if key in kw else "-"
    b = dn.index(key) + 1 if key in dn else "-"
    print(f"{key:<22}{a!s:>13}{b!s:>12}{s:>9.4f}")

## 5. Two-Stage Retrieval: Cross-Encoder Reranking
Bi-encoders embed queries and passages independently for fast Stage 1 recall (top 20). Stage 2 passes those 20 candidates alongside the query into a cross-encoder (`Xenova/ms-marco-MiniLM-L-6-v2`) to compute joint attention relevance logits, elevating the exact answer to rank 1.

In [ ]:
for question, corpus in [("can I pay the courier in cash when it arrives", "help"), ("is the Stride available in UK 9", "products")]:
    r = rs[corpus]
    print(f'\n"{question}"')
    print(" hybrid:"); show(r.hybrid(question))
    print(" rerank:"); show(r.rerank(question))

## 6. End-to-End Benchmark & Weighted Fusion Sweep
Evaluating all five retrieval strategies across 22 golden-set queries (9 exact, 13 semantic) quantifies the accuracy-latency trade-off. Reranking achieves 100% `hit@3` (22/22) and lifts MRR to 0.95 at the cost of higher per-query latency (~497 ms).

In [ ]:
print(f"{'method':<10}{'hit@3':>8}{'MRR':>7}{'exact':>8}{'meaning':>9}{'ms/q':>7}")
results = {}
for m in (*HB.METHODS, "weighted"):
    res = results[m] = HB.evaluate(rs, Q.CASES, m)
    a, e, n = res["all"], res["exact"], res["meaning"]
    print(f"{m:<10}{a['hits']:>5}/{a['n']:<2}{a['mrr']:>7.2f}{e['hits']:>5}/{e['n']:<2}{n['hits']:>6}/{n['n']:<2}{res['ms']:>7.0f}")

print("\nmisses of 'hybrid':")
for row in results["hybrid"]["rows"]:
    if row["rank"] is None:
        print(f"  {row['question']!r} wanted {row['expected']}, got {row['found']}")

**Try it:** change `alpha` in weighted fusion and watch exact vs meaning trade off:

In [ ]:
# Sweeping alpha (0.0 = pure keyword, 1.0 = pure dense) in weighted score fusion
for alpha in (0.0, 0.3, 0.5, 0.7, 1.0):
    class W:
        def __init__(self, r): self.r = r
        def search(self, m, q, k=3): return self.r.weighted(q, k, alpha=alpha)
    res = HB.evaluate({c: W(r) for c, r in rs.items()}, Q.CASES, "weighted")
    print(f"alpha {alpha:.1f}:  exact {res['exact']['hits']}/{res['exact']['n']}   meaning {res['meaning']['hits']}/{res['meaning']['n']}")

## 7. Calibrating the Relevance Gate
Retrieval engines always return a top candidate—even for out-of-scope queries (e.g., "do you sell used cars"). Auditing cross-encoder logit scores between in-scope and out-of-scope queries establishes a hard cutoff threshold, allowing the pipeline to return empty context and force a safe refusal rather than an LLM hallucination.

In [ ]:
r = rs["help"]
for q, label in [(q, "in") for q, c, *_ in Q.CASES if c == "help"][:6] + [(q, "OUT") for q, _ in Q.OUT_OF_SCOPE]:
    print(f"{r.rerank(q, k=1)[0].score:7.2f}  {label:<4} {q}")

## 8. Server-Side Hybrid Search in Qdrant
In production, both dense (384-dim) and sparse (BM25) vectors are stored per point in Qdrant. A single `query_points` request executes parallel prefetching and server-side RRF fusion, eliminating application-layer merge overhead.

In [ ]:
import vs_utils as VS
client, where = VS.connect_qdrant("auto")
name = HB.qdrant_hybrid_load(client, rs["products"])
print("loaded", name, "->", where)
for score, p in HB.qdrant_hybrid_search(client, rs["products"], "SKU-EL-004"):
    print(f"   {score:.3f}  {p['source']}  [{p['doc_id']}]")